In [0]:
%sql

create or replace table workspace.gold.hourly_cost
comment '''
    one row per home and local hour: consumption with the price that applied and the resulting energy cost. home period and subscription are matched half-open on local date; spot price is matched on utc hour. fixed tariffs use fixed_price_per_kwh, dynamic tariffs use the spot price. hours without a subscription have kwh but no price or cost. assumes ended_at is exclusive and that cost is kwh x price.
'''
as

select
    m.home_id,
    m.measured_at_local,
    m.measured_at_utc,
    h.price_area,
    s.subscription_id,
    s.tariff_type,
    coalesce(s.currency, p.currency) as currency,
    m.kwh,
    case s.tariff_type when 'fixed' then s.fixed_price_per_kwh when 'dynamic' then p.price_per_kwh end as price_per_kwh,
    m.kwh * price_per_kwh as energy_cost,
    s.monthly_fee,
    m.is_corrected,
    m.ingested_at_utc,
    current_timestamp() as _processed_at
from 
    workspace.silver.meter_readings as m
left join 
    workspace.silver.homes as h
    on h.home_id = m.home_id
    and m.measured_at_local >= h.valid_from
    and (m.measured_at_local < h.valid_to or h.valid_to is null)
left join 
    workspace.silver.subscriptions as s
    on s.home_id = m.home_id
    and m.measured_at_local >= s.started_at
    and (m.measured_at_local < s.ended_at or s.ended_at is null)
left join 
    workspace.silver.spot_prices as p
    on p.price_area = h.price_area
    and p.hour_utc = m.measured_at_utc